# Sparse prior geometry figures

This notebook rebuilds the sparse-prior visual intuition figures from the thesis text. It avoids the legacy simulation scratchpad and uses deterministic 2D and 3D toy systems only.

## Setup

The helper module in this folder owns the algebra, plotting, PyVista rendering, and figure exports. The notebook is intentionally thin so the generated figures are reproducible.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

candidate_dirs = [Path.cwd(), Path.cwd() / "notebooks" / "illustrations"]
for candidate in candidate_dirs:
    if (candidate / "sparse_prior_visuals.py").exists():
        sys.path.insert(0, str(candidate))
        ILLUSTRATION_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not find sparse_prior_visuals.py")

from sparse_prior_visuals import (
    DEFAULT_OUTPUT_DIR,
    make_2d_example,
    make_3d_example,
    plot_2d_contour,
    plot_2d_profile,
    plot_3d_unrolled,
    prior_objective,
    render_3d_isosurface,
    require_pyvista,
    validate_example,
)

OUTPUT_DIR = DEFAULT_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

In [ ]:
# PyVista is required only for the 3D isosurface figure.
try:
    pv = require_pyvista()
except ImportError as exc:
    raise ImportError("PyVista is required for the isosurface figure. Install it with: python -m pip install pyvista") from exc

print(f"PyVista {pv.__version__}")

## 2D example

The affine solution set is
$$
    x_0 + bz = \begin{bmatrix} 1.2 \\ 2.4 \end{bmatrix} + \begin{bmatrix} -2 \\ 1 \end{bmatrix}z,
$$
with sparse candidates at \(z=0.6\) and \(z=-2.4\).

In [ ]:
# Prior weights in order:
# 1. vertical ridge near x1 = 0
# 2. horizontal ridge near x2 = 0
# 3. broad central prior
#
# Lower the second weight if the contour stretches too far to the right.
# Raise the third weight to smooth the contours near the origin; lower it if it dominates.
PRIOR_WEIGHTS_2D = (1.0, 1, 0.35)

# Keep this as None for the automatic sparse-aware levels. If needed, use an explicit
# tuple such as (0.6, 0.8, 1.0, 1.2, 1.5) to hide low-value far-field contours.
CONTOUR_LEVELS_2D = None


In [ ]:
example_2d = make_2d_example(prior_weights=PRIOR_WEIGHTS_2D)
validate_example(example_2d)

print("A @ x_true =", example_2d.A @ example_2d.x_true)
print("A @ x0     =", example_2d.A @ example_2d.x0)
print("A @ B      =", example_2d.A @ example_2d.B)
print("P(x0)      =", float(prior_objective(example_2d.x0, example_2d.precisions, example_2d.prior_weights)))
for label, point in example_2d.sparse_points.items():
    print(f"P({label}) = {float(prior_objective(point, example_2d.precisions, example_2d.prior_weights)):.6f}")

In [ ]:
fig, exported_2d_contour = plot_2d_contour(
    OUTPUT_DIR,
    x1=(-2.5, 7.5, 1000),
    x2=(-2.5, 6.5, 1000),
    contour_scale="line",
    prior_weights=PRIOR_WEIGHTS_2D,
    contour_levels=CONTOUR_LEVELS_2D,
)
fig.dpi = 200
plt.show()
exported_2d_contour

In [ ]:
fig, exported_2d_profile = plot_2d_profile(OUTPUT_DIR)
plt.show()
exported_2d_profile

## 3D example

The measurements imply \(x_1+x_2+x_3=3\). The null-space coordinates use
\[
    x_0 + Bz = \begin{bmatrix}1\\1\\1\end{bmatrix}
    + \begin{bmatrix}-1 & -1\\ 1 & 0\\ 0 & 1\end{bmatrix}
      \begin{bmatrix}z_1\\z_2\end{bmatrix}.
\]
The three coordinate-sparse vertices are \((3,0,0)^\top\), \((0,3,0)^\top\), and \((0,0,3)^\top\).

In [ ]:
example_3d = make_3d_example()
validate_example(example_3d)

print("A @ x_true =", example_3d.A @ example_3d.x_true)
print("A @ x0     =", example_3d.A @ example_3d.x0)
print("A @ B      =\n", example_3d.A @ example_3d.B)
print("P(x0)      =", float(prior_objective(example_3d.x0, example_3d.precisions, example_3d.prior_weights)))
for label, point in example_3d.sparse_points.items():
    print(f"P({label}) = {float(prior_objective(point, example_3d.precisions, example_3d.prior_weights)):.6f}")

In [ ]:
exported_3d_isosurface = render_3d_isosurface(OUTPUT_DIR)
display(Image(filename=str(OUTPUT_DIR / "sparse_prior_3d_isosurface.png")))
exported_3d_isosurface

In [ ]:
fig, exported_3d_unrolled = plot_3d_unrolled(OUTPUT_DIR)
plt.show()
exported_3d_unrolled

## Export manifest

In [ ]:
exported = {
    "2d_contour": exported_2d_contour,
    "2d_profile": exported_2d_profile,
    "3d_isosurface": exported_3d_isosurface,
    "3d_unrolled": exported_3d_unrolled,
}
for name, paths in exported.items():
    print(name)
    for path in paths:
        print(" ", path)